# Dark-photon dark matter resonant conversion -- Hook, Huang & Shalaby (arXiv:2510.13956)

Run set: `pgens/dpdm` driven by `run34.sh`. Eight 1D electrostatic runs, $N_x = 1000$ over
$[0,40]\,c/\omega_p$ ($\Delta x = 0.04 \approx \lambda_D$), real mass ratio $m_i/m_e = 1836$,
$k_BT_e = k_BT_i = 10^{-3}m_ec^2$, Vay pusher, Esirkepov deposition with 5th-order splines,
`current_filters = 0`. Six amplitudes spanning $v_q^D/v_{\rm th}^e = 10^{-3}\to0.3$ at
$N_p = 2000$, plus a three-point $N_p$ convergence trio at $v_q^D/v_{\rm th}^e = 3\times10^{-3}$.

The dark photon enters as a **spatially uniform ($k=0$), time-oscillating external electric
field** $E_{\rm ext}(t) = A_0\cos(\omega t)$ acting on the Lorentz force only -- it is never a
source term in Maxwell's equations. All feedback is through the deposited current.

This is the **resonance family** ($\omega = \omega_p$ fixed). The Landau-Zener family
(paper Fig. 3) is a separate run set and is *not* reproducible from these data.

1. energy budget vs $\omega_p t$, slow and fast regimes -- **paper Fig. 2**
2. $E(k,t)$ and $n(k,t)$ spectra -- the saturation *mechanism*, **paper Fig. 8 lower**
3. $N_p$ convergence -- is the saturation physical, or shot noise?
4. regime transition across the amplitude ladder
5. $T_e/T_i(t)$ -- **paper Fig. 7**
6. real-space density cavitation -- **paper Fig. 8 upper**

**Why this run matters.** Weibel and two-stream were validation exercises: a linear growth
rate you predict analytically and then confirm. Here linear theory is only the *baseline the
data must depart from*. The paper's entire claim -- that nonlinear plasma dynamics shut off
resonant conversion and weaken DPDM limits by $3000$ to $10^7$ -- rests on **where** and
**why** the $t^2$ growth stops. Plot 1 shows that it stops; Plot 2 shows what stopped it.
Plot 3 is what makes either believable.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

try:
    plt.style.use('../../neutrino_phase_shift_dir/mine.mplstyle')
except OSError:
    pass                                  # style file optional; defaults are fine
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "text.latex.preamble": r"\usepackage{amssymb}"
})

In [ ]:
# Everything below reads the .npz written by extract_run34.py -- no nt2/adios2 needed.
NPY = "run34_npy"

meta   = json.load(open(f"{NPY}/meta.json"))
TAGS   = [k for k in meta if not k.startswith("_")]
LADDER = meta["_groups"]["ladder"]        # 6 amplitudes at ppc0=2000     -> Plot 4
CONV   = meta["_groups"]["converge"]      # ppc0 = 500 / 2000 / 8000      -> Plot 3
K      = meta["_constants"]

stats  = {t: dict(np.load(f"{NPY}/{t}_stats.npz"))  for t in TAGS}
fields = {t: dict(np.load(f"{NPY}/{t}_fields.npz")) for t in TAGS}
temps  = {t: dict(np.load(f"{NPY}/{t}_temps.npz"))  for t in TAGS}
summ   = dict(np.load(f"{NPY}/summary.npz", allow_pickle=True))

# ---- reference scales, all fixed by the setup ------------------------------
EPS_TH = K["eps_thermal_1V"]   # 5e-4 = 0.5*n_e*vth_e^2, the 1V electron thermal energy.
                               # NOT T00_1-1 = 1.5e-3, which is the 3V value.
T_ION  = K["t_ion"]            # 1/omega_pi = sqrt(1836) = 42.8, the ion response time
K_LD1  = K["k_lambdaD_1_at"]   # k*lambda_D = 1 at k = 31.6; daughters must live below this

hdr = f"{'tag':9}{'vq/vth':>9}{'A0':>11}{'ppc0':>7}{'t_sat':>8}{'eps_noise':>11}{'eps_tot(end)':>14}"
print(hdr); print("-"*len(hdr))
for t in TAGS:
    m = meta[t]
    print(f"{t:9}{m['vq_over_vthe']:>9.0e}{m['A0']:>11.3e}{int(m['ppc0']):>7}"
          f"{m['t_sat']:>8.1f}{m['eps_noise']:>11.2e}{stats[t]['eps_tot'][-1]:>14.3e}")

print(f"\nelectron thermal energy (1V) = {EPS_TH:.2e}")
print(f"ion response time 1/omega_pi = {T_ION:.1f}")
print(f"k*lambda_D = 1 at k         = {K_LD1:.1f}")
print(f"\nlinear law: {meta['_linear_law']}")

## Test 1 -- energy budget (paper Fig. 2)

The stats file is written every $1.0\,\omega_p^{-1}$ ($0.5$ for the two fastest runs) and
costs nothing; the field dumps every $10$ are far too coarse for this. As in the Weibel run,
the quantitative test lives entirely in the stats file.

**Two normalization traps, both measured (`DPDM_PIC_NOTES.md` section 0). Get either wrong
and you get a plausible-looking wrong figure.**

1. The linear-theory line goes against $\varepsilon_E + \Delta KE_e$, **not** $\varepsilon_E$
   alone. The resonantly driven Langmuir wave equipartitions, so the field carries only half
   the wave energy on cycle-average -- and instantaneously swings between 3\% and 85\% of it
   at $2\omega_p$. Measured $C = 0.12432$ against Eq. A25's $1/8 = 0.125$.
2. The thermal reference is the **1V** value $\tfrac12 n_e v_{\rm th,e}^2 = 5\times10^{-4}$,
   not $T^{00}_1 - 1 = 1.5\times10^{-3}$. Entity carries an isotropic 3V Maxwellian; SHARP is
   1V. $\Delta KE_e$ itself is safe -- with no transverse forces the $\perp$ energy is
   constant and cancels in the difference.

Grey dotted = thermal reference, crimson dotted = measured shot-noise floor
$\varepsilon_{\rm noise} = 2.06\times10^{-4}/N_p$, dash-dot vertical = $t_{\rm sat}$.

In [ ]:
def budget(ax, tag, tmin=1.0):
    s, m = stats[tag], meta[tag]
    t  = s["t"]; ok = t > tmin
    ax.loglog(t[ok], s["eps_tot"][ok], lw=2.0, c="k",
              label=r"$\varepsilon_E+\Delta KE_e$")
    ax.loglog(t[ok], s["eps_E"][ok],   lw=0.9,
              label=r"$\varepsilon_E=\frac{1}{2}\langle E_x^2\rangle$")
    ax.loglog(t[ok], np.abs(s["dKE_e"][ok]), lw=0.9, label=r"$|\Delta KE_e|$")
    ax.loglog(t[ok], np.abs(s["dKE_i"][ok]), lw=0.9, label=r"$|\Delta KE_i|$")

    # linear theory, drawn only over the range where it is supposed to hold
    tl = t[(t > tmin) & (t < 1.5*m["t_sat"])]
    ax.loglog(tl, m["A0"]**2/8*tl**2, ls="--", c="purple",
              label=r"$(A_0^2/8)(\omega_p t)^2$")

    ax.axhline(EPS_TH,          ls=":",  c="grey")
    ax.axhline(m["eps_noise"],  ls=":",  c="crimson")
    ax.axvline(m["t_sat"],      ls="-.", c="grey", lw=0.8)
    ax.set(xlabel=r"$\omega_p t$", ylabel=r"energy $[n_e m_e c^2]$",
           title=rf"{tag}:  $v_q^D/v_{{\rm th}}^e = {m['vq_over_vthe']:g}$")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.6))
budget(a1, "r3e-3")        # slow growth: saturates BELOW the thermal line
budget(a2, "r1e-1")        # fast growth: OVERSHOOTS it, then halts
a1.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show(); plt.close()

# quantify the linear law the same way the Weibel run quantified its growth rate
for tag in LADDER:
    s, m = stats[tag], meta[tag]
    fit  = (s["t"] > 0.15*m["t_sat"]) & (s["t"] < 0.5*m["t_sat"])
    Cm   = np.median(s["eps_tot"][fit]/(m["A0"]**2*s["t"][fit]**2))
    print(f"{tag:8} C = {Cm:.5f}  ({100*(Cm/0.125-1):+6.1f}% vs 1/8)   "
          f"peak eps_E/eps_th = {s['eps_E'].max()/EPS_TH:7.3f}")

**What to expect, and what is not a bug.**

- $\varepsilon_E$ alone looks like a *fuzzy band* on log axes rather than a clean line. That
  is the $2\omega_p$ equipartition swing, and it is physical. Only the black sum is smooth --
  which is exactly why the purple line must be compared to it.
- $\Delta KE_i$ is plotted as $|\Delta KE_i|$: it is tiny and can go slightly negative at
  early times while the ions equilibrate with the noise. The measured ratio
  $\Delta KE_i/\Delta KE_e = 5.70\times10^{-4}$ against $1/1836 = 5.45\times10^{-4}$ is the
  check that the drive reaches both species with the right $q/m$.
- **Slow panel** (`r3e-3`): the black curve peels away from purple and levels off *below* the
  grey thermal line. Conversion has shut down before the wave ever reaches thermal energy.
- **Fast panel** (`r1e-1`): it *overshoots* the grey line, then halts roughly
  $1/\omega_{pi} \approx 43$ after $t_{\rm sat}$, once the ions have had time to respond.

If a run's black curve sits near its crimson line rather than well above it, that run is
shot-noise dominated and should be discarded -- that is what Test 3 rules out.

## Test 2 -- $E(k,t)$ and the daughter waves (paper Fig. 8, lower)

**This is the mechanism, and it is the one plot that distinguishes "the wave saturated" from
"we know why the wave saturated."**

The $k=0$ Langmuir wave grows until its ponderomotive force
$\Phi_p(x) = e^2\hat E^2(x)/(4m_e\omega^2)$ pumps $k\neq0$ Langmuir waves (modulational
instability) and ion-acoustic waves. Those make $n_{\rm tot}(x)$ -- and hence
$\omega_p(x)$ -- non-uniform, which detunes the resonance and shuts conversion off.

The spectra are stored as $P(k) = |\mathrm{rfft}(a)/N_x|^2$ together with Parseval weights
`w_k`, so that `(w_k*P).sum() == mean(a**2)` **exactly**: 1 at $k=0$ and at Nyquist, 2
elsewhere, because `rfft` folds $\pm k$. Use them rather than rederiving the factor.

In [ ]:
def spectrum(tag, kmax=45.0):
    f, m = fields[tag], meta[tag]
    t, k, w = f["t"], f["k"], f["w_k"]
    PE   = f["PE"]
    P0   = w[0]*PE[:, 0]                        # the driven k=0 mode
    Pnz  = (w[1:]*PE[:, 1:]).sum(axis=1)        # everything it pumped
    Pn2  = (w[1:]*f["Pn2"][:, 1:]).sum(axis=1)  # ion density: ion-acoustic branch

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.6))
    sel = k <= kmax
    im = a1.pcolormesh(t, k[sel], np.log10(PE[:, sel].T + 1e-32),
                       shading="auto", cmap="magma")
    a1.axhline(K_LD1,      ls="--", c="w", lw=1.0)   # k*lambda_D = 1
    a1.axvline(m["t_sat"], ls="-.", c="w", lw=0.8)
    a1.set(xlabel=r"$\omega_p t$", ylabel=r"$k\ [\omega_p/c]$",
           title=rf"{tag}: $\log_{{10}}P_E(k,t)$")
    plt.colorbar(im, ax=a1, pad=0.02)

    a2.loglog(t, P0,  lw=1.6, label=r"$k=0$ (driven)")
    a2.loglog(t, Pnz, lw=1.6, label=r"$k\neq0$ (daughters)")
    a2.loglog(t, Pn2, lw=1.0, ls="--", label=r"$k\neq0$ of $n_i$")
    a2.axvline(m["t_sat"], ls="-.", c="grey", lw=0.8)
    a2.set(xlabel=r"$\omega_p t$", ylabel=r"$\langle \cdot^2\rangle$ by mode",
           title="where the daughters take over")
    a2.legend(fontsize=8)
    plt.tight_layout(); plt.show(); plt.close()

    xover = np.argmax(Pnz > P0)
    print(f"{tag}: k!=0 overtakes k=0 at omega_p t = "
          f"{t[xover] if xover else float('nan'):.0f}   (t_sat = {m['t_sat']:.0f})")

spectrum("r1e-1")     # fast
spectrum("r3e-3")     # slow

**What to watch.**

- Before $t_{\rm sat}$ the map is a single bright line at $k=0$ over a flat shot-noise
  background. Nothing else is happening.
- Near the departure point in Test 1, power appears at **finite $k$** -- and the crossover
  time printed above should line up with where the black curve left the purple line. That
  coincidence *is* the causal claim; if the daughters rise long after the departure,
  something else is saturating the wave.
- The daughters must live **below the white dashed line** ($k\lambda_D = 1$). Power piling up
  at the top of the box instead is a grid-scale numerical artifact, not modulational
  instability -- which is exactly why `current_filters = 0` here: 4 passes of binomial
  smoothing, as in the streaming tomls, would damp the very modes this plot is looking for.
- The dashed $n_i$ curve is the ion-acoustic branch. It should lag the electron daughters by
  something of order $1/\omega_{pi} \approx 43$, since the ions are 1836 times heavier.

## Test 3 -- $N_p$ convergence

Three runs at identical $A_0$ and $N_p = 500 / 2000 / 8000$. The measured shot-noise floor is
$\varepsilon_{\rm noise} = 2.06\times10^{-4}/N_p$, roughly $650\times$ **below** the paper's
Eq. B4 estimate -- B4 is an unscreened bare-Poisson expression and does not describe a PIC
code with Debye screening and 5th-order shape functions.

That factor is why this run set exists at $N_p \sim 10^3$ rather than the $10^7$ the analytic
budget demanded. **This plot is what makes that substitution legitimate.** If the three curves
lie on top of one another, saturation is physical and the quiet start is unnecessary.

In [ ]:
ref = stats["r3e-3"]
ok  = ref["t"] > 1.0

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.6))
for tag in CONV:
    s, m = stats[tag], meta[tag]
    lbl  = rf"$N_p={int(m['ppc0'])}$"
    a1.loglog(s["t"][s["t"] > 1.0], s["eps_tot"][s["t"] > 1.0], label=lbl)
    a1.axhline(m["eps_noise"], ls=":", lw=0.8)
    # interpolate onto the reference grid rather than assuming identical cadence
    y = np.interp(ref["t"], s["t"], s["eps_tot"])
    a2.semilogx(ref["t"][ok], y[ok]/ref["eps_tot"][ok], label=lbl)

a1.axhline(EPS_TH, ls=":", c="grey")
a1.set(xlabel=r"$\omega_p t$", ylabel=r"$\varepsilon_E+\Delta KE_e$",
       title=r"convergence at $v_q^D/v_{\rm th}^e=3\times10^{-3}$")
a1.legend(fontsize=8)
a2.axhline(1.0, ls=":", c="grey")
a2.set(xlabel=r"$\omega_p t$", ylabel=r"ratio to $N_p=2000$",
       ylim=(0.5, 1.5), title="fractional spread")
a2.legend(fontsize=8)
plt.tight_layout(); plt.show(); plt.close()

for tag in CONV:
    s, m = stats[tag], meta[tag]
    sat  = s["t"] > 1.5*m["t_sat"]
    print(f"{tag:9} Np={int(m['ppc0']):>5}  eps_noise={m['eps_noise']:.2e}  "
          f"plateau eps_tot={np.median(s['eps_tot'][sat]):.4e}  "
          f"signal/noise={np.median(s['eps_tot'][sat])/m['eps_noise']:.0f}")

**Reading the result.** The right panel is the one that matters: if the ratios sit inside a
few percent of 1 through and past saturation, the answer is $N_p$-independent and the
quiet start is not needed -- which closes **step 4** of the reproduction plan and confirms
section 3b of the notes against Eq. B4.

If instead the $N_p=500$ curve saturates high, that is shot noise seeding the daughters
prematurely, and the whole ladder would need re-running at higher $N_p$. The signal-to-noise
numbers printed above are the quick check: anything below $\sim10$ is not trustworthy.

## Test 4 -- the regime transition

The paper separates two regimes at $v_q^D/v_{\rm th}^e \sim (m_e/m_p)^{1/2}/2 \approx 0.0117$:

- **slow growth** below it -- resonance saturates *before* the Langmuir energy reaches the
  electron thermal energy;
- **fast growth** above it -- the field energy *overshoots* thermal, then halts after
  $\sim1/\omega_{pi}$ when the ions respond.

Two runs cannot demonstrate a threshold. Six can, and that is the entire reason the ladder
brackets the boundary with points on both sides.

In [ ]:
r     = summ["vq_over_vthe"]
ypk   = summ["eps_E_max_norm"]
ytot  = summ["eps_tot_max_norm"]
bnd   = float(summ["boundary"][0])

fig, ax = plt.subplots(figsize=(6.6, 4.8))
ax.loglog(r, ypk,  "o-", label=r"$\max\varepsilon_E/\varepsilon_{\rm th}$")
ax.loglog(r, ytot, "s--", label=r"$\max(\varepsilon_E+\Delta KE_e)/\varepsilon_{\rm th}$")
ax.axvline(bnd, ls="-.", c="k", label=rf"$(m_e/m_p)^{{1/2}}/2 = {bnd:.4f}$")
ax.axhline(1.0, ls=":", c="grey")
ax.set(xlabel=r"$v_q^D/v_{\rm th}^e$",
       ylabel=r"peak energy $/\ \varepsilon_{\rm th}$",
       title="slow growth saturates below 1; fast growth overshoots")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show(); plt.close()

for tag, rr, yy in zip(summ["tags"], r, ypk):
    print(f"{str(tag):8} vq/vth={rr:.0e}  peak eps_E/eps_th = {yy:8.3f}   "
          f"{'FAST (overshoot)' if yy > 1 else 'slow'}")

**The claim.** Points left of the dash-dot line should sit below the grey $y=1$ line, points
right of it above. A clean crossing near $0.0117$ reproduces the paper's regime boundary from
first principles rather than by assertion.

Expect the transition to be *gradual* rather than a step: the boundary is an order-of-magnitude
estimate, and $r = 10^{-2}$ sits essentially on top of it. A point or two straddling $y=1$ near
the line is the expected result, not a failure.

## Test 5 -- $T_e/T_i$ (paper Fig. 7)

**Parallel temperatures only.** Entity carries an isotropic 3V Maxwellian; SHARP is genuinely
1V. In a 1D electrostatic run there are no $y,z$ forces, so the perpendicular degrees of
freedom just carry constant energy -- including them would dilute $T_e$ by a factor 3 and
destroy the comparison (`DPDM_PIC_NOTES.md` section 6.5). `extract_run34.py` therefore takes
the weighted variance of $u_x$ alone.

The paper's result: in the fast regime electrons end $O(10^2)$ times hotter than they started,
with $T_e/T_i \to O(30)$ and holding steady.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.6))
for tag in ("r1e-2", "r3e-2", "r1e-1", "r3e-1"):
    T, m = temps[tag], meta[tag]
    lbl  = rf"$v_q/v_{{\rm th}}={m['vq_over_vthe']:g}$"
    a1.semilogx(T["t"], T["Te"]/T["Ti"],     label=lbl)
    a2.loglog  (T["t"], T["Te"]/T["Te"][0],  label=lbl)

a1.axhline(30,  ls=":", c="grey")
a1.set(xlabel=r"$\omega_p t$", ylabel=r"$T_e/T_i$",
       title=r"paper: $T_e/T_i \to O(30)$")
a1.legend(fontsize=8)
a2.axhline(100, ls=":", c="grey")
a2.set(xlabel=r"$\omega_p t$", ylabel=r"$T_e(t)/T_e(0)$",
       title=r"paper: electrons $O(10^2)$ times hotter")
a2.legend(fontsize=8)
plt.tight_layout(); plt.show(); plt.close()

print(f"{'tag':9}{'Te(0)':>11}{'Te(end)':>11}{'Te/Te0':>9}{'Te/Ti(end)':>12}")
for tag in LADDER:
    T = temps[tag]
    print(f"{tag:9}{T['Te'][0]:>11.3e}{T['Te'][-1]:>11.3e}"
          f"{T['Te'][-1]/T['Te'][0]:>9.1f}{T['Te'][-1]/T['Ti'][-1]:>12.1f}")

**Sanity check before believing any of it.** $T_e(0)$ should come out at
$10^{-3}\,m_ec^2$ and $T_i(0)$ at the same value -- the injector divides the TOML
`temperatures` entry by the species mass, so both species start at $k_BT = 10^{-3}m_ec^2$
even though the ions are 1836 times heavier. If $T_e(0)$ lands near $3\times10^{-3}$ instead,
the extraction picked up all three velocity components and the whole plot is wrong by 3.

The slow-growth runs should show almost no heating -- that is the point of "saturates below
the thermal energy". The separation between the slow and fast curves here is an independent
view of the same threshold as Test 4.

## Test 6 -- real-space cavitation (paper Fig. 8, upper)

The closing link in the argument. Resonance requires $\omega = \omega_p$, and
$\omega_p \propto \sqrt{n}$, so a density modulation of depth $\delta n/n$ detunes the
resonance by $\delta\omega_p/\omega_p \approx \delta n/2n$. Once that detuning exceeds the
resonance width, conversion stops -- and it stops *locally*, which is why no amount of
continued driving restarts it.

Four snapshots bracketing $t_{\rm sat}$: electron and ion density on the left axis, $E_x$ in
grey on the right.

In [ ]:
def snapshots(tag, frac=(0.5, 1.0, 2.0, 5.0)):
    f, m = fields[tag], meta[tag]
    t, x = f["t"], f["x"]
    idx  = [int(np.argmin(np.abs(t - q*m["t_sat"]))) for q in frac]

    fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
    for ax, i in zip(axes.ravel(), idx):
        ax.plot(x, f["N_1"][i], lw=0.9, label=r"$n_e$")
        ax.plot(x, f["N_2"][i], lw=0.9, label=r"$n_i$")
        ax.set(title=rf"$\omega_p t = {t[i]:.0f}$", ylabel=r"density $[n_0]$")
        axb = ax.twinx()
        axb.plot(x, f["Ex"][i], lw=0.6, c="grey", alpha=0.8)
        axb.set_ylabel(r"$E_x$", color="grey")
    for ax in axes[1]:
        ax.set_xlabel("$x\\ [c/\\omega_p]$")
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(rf"{tag}:  $v_q^D/v_{{\rm th}}^e = {m['vq_over_vthe']:g}$", y=1.01)
    plt.tight_layout(); plt.show(); plt.close()

snapshots("r1e-1")

# depth of the modulation, and the detuning it implies
print(f"{'tag':9}{'max dn/n':>11}{'at wp t':>10}{'-> dwp/wp':>12}")
for tag in LADDER:
    f  = fields[tag]
    n  = f["N_1"].astype(np.float64)
    dn = n.std(axis=1)/n.mean(axis=1)
    j  = int(dn.argmax())
    print(f"{tag:9}{dn[j]:>11.4f}{f['t'][j]:>10.0f}{0.5*dn[j]:>12.4f}")

**What to look for.** Early frames are flat to within shot noise, with $E_x$ a clean
sinusoid filling the box uniformly -- that is the $k=0$ driven mode. By $t_{\rm sat}$ the
density develops structure at the wavelengths that lit up in Test 2, and $E_x$ stops being a
single clean mode. Electrons and ions move *together* on the ion-acoustic timescale: if only
$n_e$ modulates, you are seeing the modulational instability before the ions have responded.

The printed $\delta\omega_p/\omega_p$ should be comparable to the resonance width implied by
the linear growth. That number is the quantitative version of the whole paper: it is *why*
conversion shuts off.

**Where to go next.** This run set settles the resonance family. Three things remain:

- **Landau-Zener** ($\omega$ swept slowly through $\omega_p$, paper Fig. 3) -- step 5 of the
  plan and the physically realistic case, since in cosmology it is $\omega_p(t)$ that drifts.
  The swept phase must be integrated analytically, $\phi = \omega_0 t + \tfrac12\dot\omega t^2$;
  sampling $\omega(t)$ and multiplying by $t$ deviates by up to $0.86A_0$. Already verified to
  $2.8\times10^{-14}$ in the pgen.
- **Long-term stability** to $\omega_p t = 8\times10^4$ -- the paper's strongest claim, that
  the suppressed state persists. Nothing here runs past $3000$.
- **2D cross-check** at one amplitude, to confirm the fastest modes align with the drift and
  that 1D was sufficient.

And the standing caveat: this is collisionless, like SHARP. The ion-acoustic waves damp only
on the electron-ion collision time $\tau_{ei}$, which the paper treats analytically. "Persists
forever" really means "persists on collisionless timescales."